In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
from PIL import Image

Mounted at /content/drive


In [6]:
import shutil
import os

drive_base = "/content/drive/MyDrive/Fun_projects/Detection_objets_sur_imagerie_satellite/data_lake/xView"
local_base = "/content/xView_local"

os.makedirs(local_base, exist_ok=True)

for folder in ["train_images", "yolo_labels"]:
    src = os.path.join(drive_base, folder)
    dst = os.path.join(local_base, folder)
    if not os.path.exists(dst):
        print(f"Copie de {folder}...")
        shutil.copytree(src, dst)
    else:
        print(f"{folder} déjà copié.")

print("Copie terminée.")

Copie de train_images...
Copie de yolo_labels...
Copie terminée.


In [7]:
total, used, free = shutil.disk_usage("/content")
print(f"Libre après copie : {free // (1024**3)} Go")
!du -sh /content/xView_local/*

Libre après copie : 52 Go
23G	/content/xView_local/train_images
25M	/content/xView_local/yolo_labels


In [8]:
image_dir = "/content/xView_local/train_images"
labels_dir = "/content/xView_local/yolo_labels"
output_img_dir = "/content/xView_local/tiles/images"
output_lbl_dir = "/content/xView_local/tiles/labels"

os.makedirs(output_img_dir, exist_ok=True)
os.makedirs(output_lbl_dir, exist_ok=True)

TILE_SIZE = 640
MIN_VISIBILITY = 0.3

In [9]:
def read_yolo_labels(txt_path, img_w, img_h):
    boxes = []
    if not os.path.exists(txt_path):
        return boxes
    with open(txt_path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            cls, xc, yc, w, h = map(float, line.split())
            xc, yc, w, h = xc * img_w, yc * img_h, w * img_w, h * img_h
            xmin, ymin, xmax, ymax = xc - w/2, yc - h/2, xc + w/2, yc + h/2
            boxes.append((int(cls), xmin, ymin, xmax, ymax))
    return boxes

In [10]:
def tile_image(img_id):
    img_path = os.path.join(image_dir, img_id)
    txt_path = os.path.join(labels_dir, img_id.replace('.tif', '.txt'))
    if not os.path.exists(img_path):
        return 0

    img = Image.open(img_path)
    img_w, img_h = img.size
    boxes = read_yolo_labels(txt_path, img_w, img_h)

    n_tiles_x = (img_w + TILE_SIZE - 1) // TILE_SIZE
    n_tiles_y = (img_h + TILE_SIZE - 1) // TILE_SIZE
    tiles_created = 0

    for ty in range(n_tiles_y):
        for tx in range(n_tiles_x):
            x0, y0 = tx * TILE_SIZE, ty * TILE_SIZE
            x1, y1 = min(x0 + TILE_SIZE, img_w), min(y0 + TILE_SIZE, img_h)

            tile = img.crop((x0, y0, x1, y1))
            tile_w, tile_h = tile.size
            if tile_w < 50 or tile_h < 50:
                continue

            tile_lines = []
            for cls, xmin, ymin, xmax, ymax in boxes:
                ix0, iy0 = max(xmin, x0), max(ymin, y0)
                ix1, iy1 = min(xmax, x1), min(ymax, y1)
                if ix1 <= ix0 or iy1 <= iy0:
                    continue
                orig_area = (xmax - xmin) * (ymax - ymin)
                inter_area = (ix1 - ix0) * (iy1 - iy0)
                if orig_area == 0 or inter_area / orig_area < MIN_VISIBILITY:
                    continue

                nx0, ny0, nx1, ny1 = ix0 - x0, iy0 - y0, ix1 - x0, iy1 - y0
                xc = ((nx0 + nx1) / 2) / tile_w
                yc = ((ny0 + ny1) / 2) / tile_h
                w = (nx1 - nx0) / tile_w
                h = (ny1 - ny0) / tile_h
                tile_lines.append(f"{cls} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}")

            if not tile_lines:
                continue

            base = img_id.replace('.tif', '')
            tile_name = f"{base}_{tx}_{ty}"
            tile.save(os.path.join(output_img_dir, f"{tile_name}.jpg"))
            with open(os.path.join(output_lbl_dir, f"{tile_name}.txt"), 'w') as f:
                f.write('\n'.join(tile_lines))
            tiles_created += 1

    return tiles_created

In [11]:
image_ids = [f for f in os.listdir(image_dir) if f.endswith('.tif') and not f.startswith('._')]
print(f"Nombre d'images à traiter : {len(image_ids)}")

total_tiles = 0
for i, img_id in enumerate(image_ids):
    total_tiles += tile_image(img_id)
    if i % 20 == 0:
        print(f"{i}/{len(image_ids)} images traitées, {total_tiles} tuiles générées jusqu'ici")

print(f"Tiling terminé. Total : {total_tiles} tuiles avec au moins un objet.")

Nombre d'images à traiter : 846
0/846 images traitées, 2 tuiles générées jusqu'ici
20/846 images traitées, 330 tuiles générées jusqu'ici
40/846 images traitées, 620 tuiles générées jusqu'ici
60/846 images traitées, 951 tuiles générées jusqu'ici
80/846 images traitées, 1318 tuiles générées jusqu'ici
100/846 images traitées, 1618 tuiles générées jusqu'ici
120/846 images traitées, 1846 tuiles générées jusqu'ici
140/846 images traitées, 2134 tuiles générées jusqu'ici
160/846 images traitées, 2415 tuiles générées jusqu'ici
180/846 images traitées, 2694 tuiles générées jusqu'ici
200/846 images traitées, 2942 tuiles générées jusqu'ici
220/846 images traitées, 3214 tuiles générées jusqu'ici
240/846 images traitées, 3529 tuiles générées jusqu'ici
260/846 images traitées, 3799 tuiles générées jusqu'ici
280/846 images traitées, 4045 tuiles générées jusqu'ici
300/846 images traitées, 4306 tuiles générées jusqu'ici
320/846 images traitées, 4539 tuiles générées jusqu'ici
340/846 images traitées, 483

In [12]:
n_img = len([f for f in os.listdir(output_img_dir) if f.endswith('.jpg')])
n_lbl = len([f for f in os.listdir(output_lbl_dir) if f.endswith('.txt')])
print(f"Tuiles images : {n_img}")
print(f"Tuiles labels : {n_lbl}")

Tuiles images : 12640
Tuiles labels : 12640


In [13]:
shutil.rmtree("/content/xView_local/train_images")
print("Images sources locales supprimées.")

total, used, free = shutil.disk_usage("/content")
print(f"Libre après nettoyage : {free // (1024**3)} Go")

Images sources locales supprimées.
Libre après nettoyage : 74 Go


In [15]:
dst_tiles = os.path.join(drive_base, "tiles")
if not os.path.exists(dst_tiles):
    print("Copie des tuiles vers Drive...")
    shutil.copytree(os.path.join(local_base, "tiles"), dst_tiles)
    print("Terminé.")
else:
    print("Le dossier tiles existe déjà sur Drive.")

Copie des tuiles vers Drive...
Terminé.


In [16]:
n_img_drive = len([f for f in os.listdir(os.path.join(dst_tiles, "images")) if f.endswith('.jpg')])
n_lbl_drive = len([f for f in os.listdir(os.path.join(dst_tiles, "labels")) if f.endswith('.txt')])
print(f"Sur Drive - images : {n_img_drive}, labels : {n_lbl_drive}")

Sur Drive - images : 12640, labels : 12640
